In [1]:
import pickle
import pandas as pd
import numpy as np

import easyMPRA_20231003 as em
from _helper_functions import read_tsv
from _helper_functions import revcomp
from _helper_functions import dprint

loading libraries...
done!


# Create python scripts to count barcode readcounts in RNA and DNA

In [2]:
# load file names of data to be analyzed
filedate2name={}

fn='01-Calculate-Enhancer-Activity-Levels/1-seqfile-to-samplename.tsv'
for line in read_tsv(fn,pc=False,header=False):
    if line[0][0]=='#': continue
    seqdate,fastqgz,name=line
    filedate2name[fastqgz,seqdate]=name


In [3]:
filedate2name

{('OLS_DNA_R1_S3_L001_R1_001.fastq.gz', '20210325'): 'S1_D1_L1',
 ('OLS_DNA_R1_S3_L002_R1_001.fastq.gz', '20210325'): 'S1_D1_L2',
 ('OLS_DNA_R2_S4_L001_R1_001.fastq.gz', '20210325'): 'S1_D2_L1',
 ('OLS_DNA_R2_S4_L002_R1_001.fastq.gz', '20210325'): 'S1_D2_L2',
 ('OLS_DNA_R3_S5_L003_R1_001.fastq.gz', '20210420'): 'S2_D3_L1',
 ('OLS_DNA_R3_S5_L004_R1_001.fastq.gz', '20210420'): 'S2_D3_L2',
 ('OLS_RNA_R1_S1_L001_R1_001.fastq.gz', '20210325'): 'S1_R1_L1',
 ('OLS_RNA_R1_S1_L002_R1_001.fastq.gz', '20210325'): 'S1_R1_L2',
 ('OLS_RNA_R1_S1_L003_R1_001.fastq.gz', '20210420'): 'S2_R1_L3',
 ('OLS_RNA_R1_S1_L004_R1_001.fastq.gz', '20210420'): 'S2_R1_L4',
 ('OLS_RNA_R1_S1_L001_R1_001.fastq.gz', '20210519'): 'S3_R1_L1',
 ('OLS_RNA_R1_S1_L002_R1_001.fastq.gz', '20210519'): 'S3_R1_L2',
 ('OLS_RNA_R1_S1_L003_R1_001.fastq.gz', '20210519'): 'S3_R1_L3',
 ('OLS_RNA_R1_S1_L004_R1_001.fastq.gz', '20210519'): 'S3_R1_L4',
 ('COMPLETE_OLS_RNA_R2_S2_L001_R1_001.fastq.gz', '20210325'): 'S1_R2_L1',
 ('COMPLETE_OLS_

In [4]:
# generates python script
pythonScriptTemplate=\
'''
import easyMPRA_20231003 as em

indir='00-unzipped-fastq-data/{date}/'

in_fastq = indir + "{fastqgz}".split('.gz')[0]

_=\\
em.exp_1_count_barcodes(in_fastq=in_fastq,
                        out_path="{out_path}",
                        bcLength=25,
                        out_basename="{out_basename}",
                        progressReport=1e8,
                        fastqTotalLines=None)'''

In [5]:
# generates server compute script
qsubScriptTemplate=\
'''#!/bin/bash

### The queue (-q) to which the job is being submitted to:
### *condo is the queue for when we need more resources than we own,
### *glean is the free queue based on open resources on TSCC,
### *home  is our home queue that only farley lab members can use
### The account (-A) will always be farley-group
#PBS -q condo
#PBS -A farley-group

### Specify the name of the notebook job, output file, and error/log file;
### $PBS_JOBID embeds the job id in the script (helpful for debugging, but can be removed)
#PBS -N analyze-barcodes
#PBS -o {pyscript}.log
#PBS -e {pyscript}.err

### Request resources for the job, and the walltime for the job to run
### (note: ppn - processors per node; maximum ppn is 16)
### Generally try to run jobs with a single node and increase ppn
#PBS -l nodes=1:ppn=32
#PBS -l walltime=8:00:00

##### Exports all user environment variables to the job
#PBS -V

### Specify the email (can be multiple, separated by commas) to recieve
### emails/diagnostic information about the job; -m sets the mail settings:
### (n: no mail sent, a: mail sent when job aborted, b: mail sent when job begins,
### e: mail sent when job terminates)
#PBS -M solvason@eng.ucsd.edu
#PBS -m abe

#######################################################
# !!!!! DIAGNOSTIC STATEMENTS BELOW THIS HEADER !!!!! #
#######################################################

# Print out diagnostic information to have within the output file,
# This information can be removed or kept as desired
echo "Diagnostic Information:"
echo $PBS_JOBID
echo $PBS_JOBNAME
echo $PBS_NODELIST
echo $PBS_O_WORKDIR
echo $PBS_USER
echo ""

echo "Connected Login Node:"
echo $PBS_O_HOST
echo ""

##############################################
# !!!!! JOB COMMANDS BELOW THIS HEADER !!!!! #
##############################################

date

source ~/.bash_profile
source ~/.condainit
conda activate notebook

python {pyscript}

date
'''

In [6]:
##############################################
# generate batch scripts for each sample
##############################################

out_path='01-Calculate-Enhancer-Activity-Levels/01-table-processing/'
Sample2Bc2ReadCount={}


for (fastqgz,date),sampleName in filedate2name.items():

    print(sampleName,date)
    
    # python script
    outname  =f'{out_path}/{sampleName}.py'
    outscript=pythonScriptTemplate.replace('{date}',date).replace('{fastqgz}',fastqgz).replace('{sampleName}',sampleName).replace('{out_path}',out_path).replace('{out_basename}',sampleName)
    with open(outname,'w') as f: f.write(outscript)
    
    # qsub script 
    outqsub  =f'{out_path}/GO.{sampleName}'
    outqsubscript=qsubScriptTemplate.replace('{pyscript}',outname)
    with open(outqsub,'w') as f: f.write(outqsubscript)


    # break

S1_D1_L1 20210325
S1_D1_L2 20210325
S1_D2_L1 20210325
S1_D2_L2 20210325
S2_D3_L1 20210420
S2_D3_L2 20210420
S1_R1_L1 20210325
S1_R1_L2 20210325
S2_R1_L3 20210420
S2_R1_L4 20210420
S3_R1_L1 20210519
S3_R1_L2 20210519
S3_R1_L3 20210519
S3_R1_L4 20210519
S1_R2_L1 20210325
S1_R2_L2 20210325
S2_R2_L1 20210420
S2_R2_L2 20210420
S3_R2_L1 20210519
S3_R2_L2 20210519
S3_R2_L3 20210519
S3_R2_L4 20210519
S2_R3_L1 20210420
S2_R3_L2 20210420


# Produce barcode readcount table

In [7]:
# filedate2name

In [7]:
# the first time you run this, combine all readcount files into a single dictionary and then save it


# datadir='01-Calculate-Enhancer-Activity-Levels/01-table-processing/'

# Sample2Bc2ReadCount={}

# for (file,seqdate),name in filedate2name.items():

#     if name in Sample2Bc2ReadCount:
#         print('Sample',name,'already detected... skipping...')
        
#     else:
#         fn=f'{datadir}/{name}.Bc2ReadCount.pickle'        
#         try:
#             print('Reading',name,'...')
#             with open(fn,'rb') as f: Bc2ReadCount=pickle.load(f)
#             Sample2Bc2ReadCount[name]=Bc2ReadCount
            
#             # print('detected! ',name)
#             # with open(fn,'rb') as f: pass
            
#         except FileNotFoundError:
#             print('\nFILE NOT FOUND',fn)
#             print()
            
# del Bc2ReadCount

Reading S1_D1_L1 ...
Reading S1_D1_L2 ...
Reading S1_D2_L1 ...
Reading S1_D2_L2 ...
Reading S2_D3_L1 ...
Reading S2_D3_L2 ...
Reading S1_R1_L1 ...
Reading S1_R1_L2 ...
Reading S2_R1_L3 ...
Reading S2_R1_L4 ...
Reading S3_R1_L1 ...
Reading S3_R1_L2 ...
Reading S3_R1_L3 ...
Reading S3_R1_L4 ...
Reading S1_R2_L1 ...
Reading S1_R2_L2 ...
Reading S2_R2_L1 ...
Reading S2_R2_L2 ...
Reading S3_R2_L1 ...
Reading S3_R2_L2 ...
Reading S3_R2_L3 ...
Reading S3_R2_L4 ...
Reading S2_R3_L1 ...
Reading S2_R3_L2 ...


In [8]:
with open('01-Calculate-Enhancer-Activity-Levels/01-table-processing/20231005-combined-Sample2Bc2ReadCount.pydict.pickle','wb') as f:
    pickle.dump(Sample2Bc2ReadCount,f,protocol=pickle.HIGHEST_PROTOCOL)

# read the file
# fn='01-Calculate-Enhancer-Activity-Levels/01-table-processing/20231005-combined-Sample2Bc2ReadCount.pydict.pickle'
# with open(fn,'rb') as f: Sample2Bc2ReadCount=pickle.load(f)


In [9]:
def io_ubc2en_from_tsv(ubc2en_tsv):
    '''Converts ubc2en (unique barcode to enhancer) from tsv to python dictionary.'''
    ubc2en={}
    for ubc,en in io_yield_tsvRows(ubc2en_tsv,pc=False,header=False):
        if ubc not in ubc2en:
            ubc2en[ubc]={}
        ubc2en[ubc]=en
    return ubc2en
    
def io_yield_tsvRows(fn,pc,header,breakBool=False,sep='\t',pc_list=False):
    '''Read a tsv file'''
    with open(fn,'r') as f:

        # If printing columns, skip header
        if header:
            if (pc==True): pass
            else:          next(f)

        for i,line in enumerate(f):
            a=line.strip().split(sep)
            if pc:
                if pc_list==False:
                    if i==0:
                        for i,c in enumerate(a):
                            print(i,c)
                        print()
                        if breakBool: break
                        continue
                else:
                    if i==0:
                        print(', '.join([i.replace('-','_').replace(' ','_') for i in a]))
                        if breakBool: break
                        continue
            yield a
            
def io_enhancerOrderSheet_from_tsv(enOrderSheet):
    
    EnOrderName2Seq={}
    EnOrderSeq2Name={}
    for row in io_yield_tsvRows(enOrderSheet,pc=False,header=True):
        name,seq=row
        # seq=seq[25:-25] # delete [25:-25] if no sequencing adapters a re in the order sheet
        EnOrderName2Seq[name]=seq
        EnOrderSeq2Name[seq ]=name
        
    return EnOrderName2Seq,EnOrderSeq2Name


In [10]:
# read in dictionary 

minreads=1
fn='../0-produce-en-bc-dict/01c-dicts/3-filt-dict_ubc2en_filt_minReads=1.tsv'

# dictname=f'2023-reanalysis-minreads={minreads}'
ubc2en=io_ubc2en_from_tsv(fn)

ubc2en={ubc[:25]:en for ubc,en in ubc2en.items()}


In [11]:
!head {enOrderSheet}

head: cannot open '{enOrderSheet}' for reading: No such file or directory


In [12]:
enOrderSheet='../0-produce-en-bc-dict/otxa-scrambled-wt-and-ablated.tsv'

In [13]:
_,En2EnId=io_enhancerOrderSheet_from_tsv(enOrderSheet)

In [14]:
dprint(En2EnId)

CATCTGAAGCTCGTTATCTCTAACGGAAGTTTTCGAAAAGGAAATTGCTCAATATCTAAGATAGGA L1-G1R-L2-E1F-L3-E2F-L4-G2R-L5-G3F-L6


In [15]:

bcdf=\
em.exp_2_create_barcode_table(Sample2Bc2ReadCount, 
                              enOrderSheet,
                              ubc2en)


Initializing barcode table with dictionary sequences...

	36,082,876 barcodes detected in the dictionary

Adding barcode readcounts from S1_D1_L1...

        Number Barcodes Analyzed       = 92,046,229
        Number Barcodes in Dictionary  = 34,514,503 (37.5%)
        Number Reads Analyzed          = 2,072,856,190
        Number Reads in Dict Barcodes  = 755,318,905 (36.4%)
        

Adding barcode readcounts from S1_D1_L2...

        Number Barcodes Analyzed       = 91,417,618
        Number Barcodes in Dictionary  = 34,525,269 (37.8%)
        Number Reads Analyzed          = 2,115,628,748
        Number Reads in Dict Barcodes  = 771,251,936 (36.5%)
        

Adding barcode readcounts from S1_D2_L1...

        Number Barcodes Analyzed       = 82,294,651
        Number Barcodes in Dictionary  = 33,210,520 (40.4%)
        Number Reads Analyzed          = 1,545,359,445
        Number Reads in Dict Barcodes  = 563,342,809 (36.5%)
        

Adding barcode readcounts from S1_D2_L2...

    

In [16]:
bcdf.head()

,barcode-seq,enhancer-seq,enhancer-id,rc-S1_D1_L1,rpm-S1_D1_L1,rc-S1_D1_L2,rpm-S1_D1_L2,rc-S1_D2_L1,rpm-S1_D2_L1,rc-S1_D2_L2,rpm-S1_D2_L2,rc-S2_D3_L1,rpm-S2_D3_L1,rc-S2_D3_L2,rpm-S2_D3_L2,rc-S1_R1_L1,rpm-S1_R1_L1,rc-S1_R1_L2,rpm-S1_R1_L2,rc-S2_R1_L3,rpm-S2_R1_L3,rc-S2_R1_L4,rpm-S2_R1_L4,rc-S3_R1_L1,rpm-S3_R1_L1,rc-S3_R1_L2,rpm-S3_R1_L2,rc-S3_R1_L3,rpm-S3_R1_L3,rc-S3_R1_L4,rpm-S3_R1_L4,rc-S1_R2_L1,rpm-S1_R2_L1,rc-S1_R2_L2,rpm-S1_R2_L2,rc-S2_R2_L1,rpm-S2_R2_L1,rc-S2_R2_L2,rpm-S2_R2_L2,rc-S3_R2_L1,rpm-S3_R2_L1,rc-S3_R2_L2,rpm-S3_R2_L2,rc-S3_R2_L3,rpm-S3_R2_L3,rc-S3_R2_L4,rpm-S3_R2_L4,rc-S2_R3_L1,rpm-S2_R3_L1,rc-S2_R3_L2,rpm-S2_R3_L2
0,CTTGCCACAGCAGCTCCCTGTCTAA,ACTTCCGTTAGTTATCTCTTTCGAAAAGGAAATCATCTGAAGCTCA...,L5-E1R-L2-G1R-L3-E2F-L1-G2R-L4-G3F-L6,37.0,0.028083,35.0,0.026034,3.0,0.003055,12.0,0.012078,20.0,0.032257,24.0,0.039091,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ACCCAGGTTAATTTCCGCGCTTGAC,TAAGATATTTGCTCATTTCCTTACGGAAGTCATCTGAAGCTCCCTA...,L2-G2F-L4-E2R-L5-E1F-L1-G3R-L3-G1F-L6,25.0,0.018975,25.0,0.018596,14.0,0.014256,12.0,0.012078,14.0,0.022580,12.0,0.019545,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AAGGGTGTAGTAAAAAGTTATAATA,TGCTCGAAATAACTTTCGAAATTTGCTTTAAATATTTGCTATTTTC...,ABL_L4-G1F-L3-E2R-L2-G2R-L5-G3R-L1-E1F-L6,1.0,0.000759,6.0,0.004463,7.0,0.007128,4.0,0.004026,2.0,0.003226,1.0,0.001629,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,TCCCGGCGAGCTGGTGTGTCTCAGA,TGCTCAATATCTTAGTTATCTCTTTCGAAACGGAAGTCATCTGAAG...,L4-G2R-L2-G1R-L3-E1F-L1-G3R-L5-E2F-L6,13.0,0.009867,6.0,0.004463,6.0,0.006110,8.0,0.008052,1.0,0.001613,1.0,0.001629,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ATATGTTACGCGGAATAGCTCGCCC,TAAAATATTTGCTCATTTGCTTTTTCGAAACTTGCGTGAAATAACC...,ABL_L2-G2F-L4-E2R-L3-E1R-L5-G1F-L1-G3R-L6,42.0,0.031878,35.0,0.026034,4.0,0.004073,2.0,0.002013,20.0,0.032257,12.0,0.019545,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,206.0,0.335147,231.0,0.37696


In [ ]:
# if writing
bcdf.to_csv('01-Calculate-Enhancer-Activity-Levels/02-generated-tables/bcdf.tsv',sep='\t',index=None)
bcdf.to_pickle('01-Calculate-Enhancer-Activity-Levels/02-generated-tables/bcdf.pd.df.pickle')

In [2]:
# if reading
bcdf=pd.read_pickle('01-Calculate-Enhancer-Activity-Levels/02-generated-tables/bcdf.pd.df.pickle')

In [3]:
bcdf['enhancer-id']=bcdf['enhancer-id'].apply(lambda n: n.replace('S','L'))

In [4]:
bcdf.head()

,barcode-seq,enhancer-seq,enhancer-id,rc-S1_D1_L1,rpm-S1_D1_L1,rc-S1_D1_L2,rpm-S1_D1_L2,rc-S1_D2_L1,rpm-S1_D2_L1,rc-S1_D2_L2,...,rc-S3_R2_L2,rpm-S3_R2_L2,rc-S3_R2_L3,rpm-S3_R2_L3,rc-S3_R2_L4,rpm-S3_R2_L4,rc-S2_R3_L1,rpm-S2_R3_L1,rc-S2_R3_L2,rpm-S2_R3_L2
0,CTTGCCACAGCAGCTCCCTGTCTAA,ACTTCCGTTAGTTATCTCTTTCGAAAAGGAAATCATCTGAAGCTCA...,L5-E1R-L2-G1R-L3-E2F-L1-G2R-L4-G3F-L6,37.0,0.028083,35.0,0.026034,3.0,0.003055,12.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ACCCAGGTTAATTTCCGCGCTTGAC,TAAGATATTTGCTCATTTCCTTACGGAAGTCATCTGAAGCTCCCTA...,L2-G2F-L4-E2R-L5-E1F-L1-G3R-L3-G1F-L6,25.0,0.018975,25.0,0.018596,14.0,0.014256,12.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AAGGGTGTAGTAAAAAGTTATAATA,TGCTCGAAATAACTTTCGAAATTTGCTTTAAATATTTGCTATTTTC...,ABL_L4-G1F-L3-E2R-L2-G2R-L5-G3R-L1-E1F-L6,1.0,0.000759,6.0,0.004463,7.0,0.007128,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,TCCCGGCGAGCTGGTGTGTCTCAGA,TGCTCAATATCTTAGTTATCTCTTTCGAAACGGAAGTCATCTGAAG...,L4-G2R-L2-G1R-L3-E1F-L1-G3R-L5-E2F-L6,13.0,0.009867,6.0,0.004463,6.0,0.006110,8.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ATATGTTACGCGGAATAGCTCGCCC,TAAAATATTTGCTCATTTGCTTTTTCGAAACTTGCGTGAAATAACC...,ABL_L2-G2F-L4-E2R-L3-E1R-L5-G1F-L1-G3R-L6,42.0,0.031878,35.0,0.026034,4.0,0.004073,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,206.0,0.335147,231.0,0.37696


In [5]:
# only keep rpm columns
for c in [i for i in bcdf.columns if 'rc-' in i]:
    print(c)
    bcdf=bcdf.drop(c,axis=1)

rc-S1_D1_L1
rc-S1_D1_L2
rc-S1_D2_L1
rc-S1_D2_L2
rc-S2_D3_L1
rc-S2_D3_L2
rc-S1_R1_L1
rc-S1_R1_L2
rc-S2_R1_L3
rc-S2_R1_L4
rc-S3_R1_L1
rc-S3_R1_L2
rc-S3_R1_L3
rc-S3_R1_L4
rc-S1_R2_L1
rc-S1_R2_L2
rc-S2_R2_L1
rc-S2_R2_L2
rc-S3_R2_L1
rc-S3_R2_L2
rc-S3_R2_L3
rc-S3_R2_L4
rc-S2_R3_L1
rc-S2_R3_L2


# Pool data from all replicates 

## take averaged readcount across lanes

In [6]:

SampleScaffoldKey2ValueList={
    '{nt}':['R','D'],
    '{rep}':[1,2,3],
    '{seqrun}':[1,2,3],
    '{lane}':[1,2,3,4],
}


In [7]:

# within each fraction
for nt in ['R','D']:
    
    # within each rep
    for rep in ['1','2','3']:

        # within each sequencing run
        for seqrun in ['1','2','3']:

            print()
            comparisonString='rpm-S{seqrun}_{nt}{rep}_L{lane}'.replace('{nt}',nt).replace('{seqrun}',seqrun).replace('{rep}',rep)
            comparisonList=\
            em.label_generator( comparisonString,
                                SampleScaffoldKey2ValueList,
                                '{lane}',
                                bcdf.columns)

            if len(comparisonList)<=1: continue

            concatcol=f'rpm-S{seqrun}_{nt}{rep}_L-concat'
            aggcol=f'rpm-S{seqrun}_{nt}{rep}_L-average'
            
            print(comparisonList)
            print(aggcol)

            # if aggcol in bcdf: continue

            bcdf[aggcol]=bcdf[comparisonList].mean(axis=1, skipna=True)

            # bcdf[concatcol]=em.concat_sample_values(indf=bcdf,colList=comparisonList)        
            # bcdf[aggcol]=em.aggregate_within_samples(indf=bcdf,col=concatcol,aggfunc=np.mean)
            # bcdf=bcdf.drop(concatcol,axis=1)

bcdf.head()


[WARNING] column rpm-S1_R1_L3 not in dataframe
[WARNING] column rpm-S1_R1_L4 not in dataframe
['rpm-S1_R1_L1', 'rpm-S1_R1_L2']
rpm-S1_R1_L-average

[WARNING] column rpm-S2_R1_L1 not in dataframe
[WARNING] column rpm-S2_R1_L2 not in dataframe
['rpm-S2_R1_L3', 'rpm-S2_R1_L4']
rpm-S2_R1_L-average

['rpm-S3_R1_L1', 'rpm-S3_R1_L2', 'rpm-S3_R1_L3', 'rpm-S3_R1_L4']
rpm-S3_R1_L-average

[WARNING] column rpm-S1_R2_L3 not in dataframe
[WARNING] column rpm-S1_R2_L4 not in dataframe
['rpm-S1_R2_L1', 'rpm-S1_R2_L2']
rpm-S1_R2_L-average

[WARNING] column rpm-S2_R2_L3 not in dataframe
[WARNING] column rpm-S2_R2_L4 not in dataframe
['rpm-S2_R2_L1', 'rpm-S2_R2_L2']
rpm-S2_R2_L-average

['rpm-S3_R2_L1', 'rpm-S3_R2_L2', 'rpm-S3_R2_L3', 'rpm-S3_R2_L4']
rpm-S3_R2_L-average

[WARNING] column rpm-S1_R3_L1 not in dataframe
[WARNING] column rpm-S1_R3_L2 not in dataframe
[WARNING] column rpm-S1_R3_L3 not in dataframe
[WARNING] column rpm-S1_R3_L4 not in dataframe

[WARNING] column rpm-S2_R3_L3 not in dataframe

,barcode-seq,enhancer-seq,enhancer-id,rpm-S1_D1_L1,rpm-S1_D1_L2,rpm-S1_D2_L1,rpm-S1_D2_L2,rpm-S2_D3_L1,rpm-S2_D3_L2,rpm-S1_R1_L1,...,rpm-S1_R1_L-average,rpm-S2_R1_L-average,rpm-S3_R1_L-average,rpm-S1_R2_L-average,rpm-S2_R2_L-average,rpm-S3_R2_L-average,rpm-S2_R3_L-average,rpm-S1_D1_L-average,rpm-S1_D2_L-average,rpm-S2_D3_L-average
0,CTTGCCACAGCAGCTCCCTGTCTAA,ACTTCCGTTAGTTATCTCTTTCGAAAAGGAAATCATCTGAAGCTCA...,L5-E1R-L2-G1R-L3-E2F-L1-G2R-L4-G3F-L6,0.028083,0.026034,0.003055,0.012078,0.032257,0.039091,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.027059,0.007566,0.035674
1,ACCCAGGTTAATTTCCGCGCTTGAC,TAAGATATTTGCTCATTTCCTTACGGAAGTCATCTGAAGCTCCCTA...,L2-G2F-L4-E2R-L5-E1F-L1-G3R-L3-G1F-L6,0.018975,0.018596,0.014256,0.012078,0.022580,0.019545,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.018785,0.013167,0.021063
2,AAGGGTGTAGTAAAAAGTTATAATA,TGCTCGAAATAACTTTCGAAATTTGCTTTAAATATTTGCTATTTTC...,ABL_L4-G1F-L3-E2R-L2-G2R-L5-G3R-L1-E1F-L6,0.000759,0.004463,0.007128,0.004026,0.003226,0.001629,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.002611,0.005577,0.002427
3,TCCCGGCGAGCTGGTGTGTCTCAGA,TGCTCAATATCTTAGTTATCTCTTTCGAAACGGAAGTCATCTGAAG...,L4-G2R-L2-G1R-L3-E1F-L1-G3R-L5-E2F-L6,0.009867,0.004463,0.006110,0.008052,0.001613,0.001629,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.007165,0.007081,0.001621
4,ATATGTTACGCGGAATAGCTCGCCC,TAAAATATTTGCTCATTTGCTTTTTCGAAACTTGCGTGAAATAACC...,ABL_L2-G2F-L4-E2R-L3-E1R-L5-G1F-L1-G3R-L6,0.031878,0.026034,0.004073,0.002013,0.032257,0.019545,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.356054,0.028956,0.003043,0.025901


## averaging readcounts across sequencing runns

In [8]:
keepCols=[c for c in bcdf.columns if 'L-average' in c or 'enhancer' in c or 'barcode' in c]
# keepCols
bcdf=bcdf.loc[:,keepCols]

In [ ]:

for nt in ['R','D']:
    
    # within each rep
    for rep in ['1','2','3']:

        print()
        comparisonString='rpm-S{seqrun}_{nt}{rep}_L-average'.replace('{nt}',nt).replace('{rep}',rep)
        comparisonList=\
        em.label_generator( comparisonString,
                            SampleScaffoldKey2ValueList,
                            '{seqrun}',
                            bcdf.columns)

        # if len(comparisonList)<=1: continue

        aggcol=f'rpm-S-average_{nt}{rep}_L-average'
        
        print(comparisonList)
        print(aggcol)

        if aggcol in bcdf: continue

        bcdf[aggcol]=bcdf[comparisonList].mean(axis=1, skipna=True)

bcdf.head()


['rpm-S1_R1_L-average', 'rpm-S2_R1_L-average', 'rpm-S3_R1_L-average']
rpm-S-average_R1_L-average

['rpm-S1_R2_L-average', 'rpm-S2_R2_L-average', 'rpm-S3_R2_L-average']
rpm-S-average_R2_L-average


# Keep only the final RPM values for each barcode

In [ ]:
keepCols=[c for c in bcdf.columns if 'S-average' in c or 'enhancer' in c or 'barcode' in c]
# keepCols
bcdf=bcdf.loc[:,keepCols]

In [ ]:
bcdf.head()

# Produce enhancer table

In [ ]:
endf=em.exp_3_bcdf_to_endf(bcdf)
endf.head()

In [28]:
# simplify names
endf.columns=[c.replace('S-average_','').replace('_L-average','') for c in endf.columns]
endf.head()

,enhancer-seq,enhancer-id,barcode-seq,rpm-R1,rpm-R2,rpm-R3,rpm-D1,rpm-D2,rpm-D3
0,AAAATAGGCATCTGAAGCTCAAATATTTAAAGCAAATTGCTCACGC...,ABL_L5-G3F-L1-G2F-L2-E2F-L4-E1F-L3-G1F-L6,"[AATCACGCTAGTAATGATCCTGTCA, TGCTCAGGATCTCATGCT...","[nan, 0.2155436763930323, nan, nan, nan, nan, ...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[nan, nan, 0.39613400723581615, nan, nan, nan,...","[0.03360226480751837, 0.006921601131334899, 0....","[0.025046072964932543, 0.0073385783122074996, ...","[0.016504956489254687, 0.010180072101118692, 0..."
1,AAAATAGGCATCTGAAGCTCAAATATTTAAAGCAAATTGCTCACGC...,ABL_L5-G3F-L1-G2F-L2-E2F-L4-E1F-L3-G1R-L6,"[CCCTCGCGAGCAAATCAACTTAATC, CTTGCCGCTAGTGTGAAT...","[nan, nan, nan, nan, nan, nan, nan, 0.46241677...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[0.016490982509852568, 0.03270873716764157, 0....","[0.017720906546900027, 0.03252115663104209, 0....","[0.021252142695286635, 0.025671444566973223, 0..."
2,AAAATAGGCATCTGAAGCTCAAATATTTAAAGCAAATTGCTCACTT...,ABL_L5-G3F-L1-G2F-L2-E2F-L4-E1R-L3-G1F-L6,"[TCAATTGCTTCCAAATGACAGCGCC, AGCTAAATGGGCGATATG...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[0.045958770309293656, 0.018622306219179978, 0...","[0.007692037736651464, 0.016547098739350063, 0...","[0.04544642878023401, 0.015159773114774597, 0...."
3,AAAATAGGCATCTGAAGCTCAAATATTTAAAGCAAATTGCTCACTT...,ABL_L5-G3F-L1-G2F-L2-E2F-L4-E1R-L3-G1R-L6,"[AGTGCACTTAGAACAACGGCAAATG, TCAGTAACATGGCTTCCG...","[nan, nan, nan, nan, nan, nan, nan, 0.33541042...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[0.03744823878430403, 0.0022404604475985994, 0...","[0.01972557028633456, 0.006043073732577989, 0....","[0.04844621293815733, 0.001618961699054195, 0...."
4,AAAATAGGCATCTGAAGCTCAAATATTTAAAGCAAATTGCTCGAAA...,ABL_L5-G3F-L1-G2F-L2-E2F-L4-G1F-L3-E1F-L6,"[ATAACGATACCCGAGCAAAATATTT, TTAGAACATTAGCACACA...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[nan, nan, nan, nan, nan, nan, nan, nan, 0.468...","[0.004343002705638193, 0.02633657053323679, 0....","[0.008273512291925687, 0.02799092817134153, 0....","[0.007832628007784115, 0.03226342555573737, 0...."


In [29]:
# simplify names
bcdf.columns=[c.replace('S-average_','').replace('_L-average','') for c in bcdf.columns]
bcdf.head()

,barcode-seq,enhancer-seq,enhancer-id,rpm-R1,rpm-R2,rpm-R3,rpm-D1,rpm-D2,rpm-D3
0,CTTGCCACAGCAGCTCCCTGTCTAA,ACTTCCGTTAGTTATCTCTTTCGAAAAGGAAATCATCTGAAGCTCA...,L5-E1R-L2-G1R-L3-E2F-L1-G2R-L4-G3F-L6,NaN,NaN,NaN,0.027059,0.007566,0.035674
1,ACCCAGGTTAATTTCCGCGCTTGAC,TAAGATATTTGCTCATTTCCTTACGGAAGTCATCTGAAGCTCCCTA...,L2-G2F-L4-E2R-L5-E1F-L1-G3R-L3-G1F-L6,NaN,NaN,NaN,0.018785,0.013167,0.021063
2,AAGGGTGTAGTAAAAAGTTATAATA,TGCTCGAAATAACTTTCGAAATTTGCTTTAAATATTTGCTATTTTC...,ABL_L4-G1F-L3-E2R-L2-G2R-L5-G3R-L1-E1F-L6,NaN,NaN,NaN,0.002611,0.005577,0.002427
3,TCCCGGCGAGCTGGTGTGTCTCAGA,TGCTCAATATCTTAGTTATCTCTTTCGAAACGGAAGTCATCTGAAG...,L4-G2R-L2-G1R-L3-E1F-L1-G3R-L5-E2F-L6,NaN,NaN,NaN,0.007165,0.007081,0.001621
4,ATATGTTACGCGGAATAGCTCGCCC,TAAAATATTTGCTCATTTGCTTTTTCGAAACTTGCGTGAAATAACC...,ABL_L2-G2F-L4-E2R-L3-E1R-L5-G1F-L1-G3R-L6,NaN,NaN,0.356054,0.028956,0.003043,0.025901


In [30]:
endf.to_pickle(f'01-Calculate-Enhancer-Activity-Levels/02-generated-tables/endf.rpmOnly.laneAvg.seqrunAvg.pd.df.pickle')
bcdf.to_pickle(f'01-Calculate-Enhancer-Activity-Levels/02-generated-tables/bcdf.rpmOnly.laneAvg.seqrunAvg.pd.df.pickle')